In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
patients = pd.read_csv("patients_cleaned.csv")

medicine = pd.read_csv("medicine_cleaned.csv")

In [5]:
patients.head()


,Patient_ID,Patient_Name,Age,Gender,Village,Visit_Date,Season,Disease,Medicine_Name,Quantity,Doctor,Year,Month
0,1,Patient_1,4,Female,Sonapur,2025-12-01,Winter,Fever,Paracetamol,1,Dr. Sharma,2025,12
1,2,Patient_2,64,Female,Lakshmi Nagar,2025-11-25,Post-Monsoon,Diarrhoea,ORS,10,Dr. Deshmukh,2025,11
2,3,Patient_3,2,Male,Rahatgaon,2026-05-19,Summer,Cough,Cough Syrup,1,Dr. Deshmukh,2026,5
3,4,Patient_4,41,Male,Bhavanipur,2025-05-26,Summer,Common Cold,Cetirizine,10,Dr. Singh,2025,5
4,5,Patient_5,33,Male,Khed,2025-03-12,Summer,Anaemia,Iron Tablets,3,Dr. Patil,2025,3


In [6]:
medicine.head()

,Medicine_ID,Medicine_Name,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name
0,M001,Paracetamol,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC
1,M002,ORS,180,80,In Stock,2027-03-15,HealthMed,Rampur PHC
2,M003,Cough Syrup,75,50,Low Stock,2026-11-10,MediCare,Shivpur PHC
3,M004,Cetirizine,220,60,In Stock,2027-01-20,ABC Pharma,Shivpur PHC
4,M005,Amoxicillin,95,100,Low Stock,2026-12-25,LifeCare,Ganeshwadi PHC


In [7]:
medicine_usage = (

    patients

    .groupby(

        ["Disease","Medicine_Name"]

    )["Quantity"]

    .sum()

    .reset_index()

)

medicine_usage

,Disease,Medicine_Name,Quantity
0,Anaemia,Iron Tablets,1565
1,Asthma,Salbutamol,1605
2,Common Cold,Cetirizine,1614
3,Cough,Cough Syrup,1302
4,Dengue,Paracetamol,1479
5,Diarrhoea,ORS,1547
6,Fever,Paracetamol,1517
7,Malaria,Artemether,1483
8,Pneumonia,Amoxicillin,1389
9,TB,Rifampicin,1374


In [8]:
patient_count = (

    patients

    .groupby("Disease")

    .size()

    .reset_index(name="Patients")

)

patient_count

,Disease,Patients
0,Anaemia,276
1,Asthma,290
2,Common Cold,284
3,Cough,240
4,Dengue,274
5,Diarrhoea,288
6,Fever,268
7,Malaria,282
8,Pneumonia,258
9,TB,270


In [9]:
medicine_usage = medicine_usage.merge(

    patient_count,

    on="Disease"

)

medicine_usage

,Disease,Medicine_Name,Quantity,Patients
0,Anaemia,Iron Tablets,1565,276
1,Asthma,Salbutamol,1605,290
2,Common Cold,Cetirizine,1614,284
3,Cough,Cough Syrup,1302,240
4,Dengue,Paracetamol,1479,274
5,Diarrhoea,ORS,1547,288
6,Fever,Paracetamol,1517,268
7,Malaria,Artemether,1483,282
8,Pneumonia,Amoxicillin,1389,258
9,TB,Rifampicin,1374,270


In [10]:
medicine_usage["Medicine_Per_Patient"] = (

    medicine_usage["Quantity"]

    /

    medicine_usage["Patients"]

)

medicine_usage

,Disease,Medicine_Name,Quantity,Patients,Medicine_Per_Patient
0,Anaemia,Iron Tablets,1565,276,5.670290
1,Asthma,Salbutamol,1605,290,5.534483
2,Common Cold,Cetirizine,1614,284,5.683099
3,Cough,Cough Syrup,1302,240,5.425000
4,Dengue,Paracetamol,1479,274,5.397810
5,Diarrhoea,ORS,1547,288,5.371528
6,Fever,Paracetamol,1517,268,5.660448
7,Malaria,Artemether,1483,282,5.258865
8,Pneumonia,Amoxicillin,1389,258,5.383721
9,TB,Rifampicin,1374,270,5.088889


In [11]:
forecast = pd.read_csv("disease_forecast.csv")

In [12]:
required = forecast.merge(

    medicine_usage,

    on="Disease"

)

required

,Village,Disease,Season,Year,Month,Predicted_Cases,Medicine_Name,Quantity,Patients,Medicine_Per_Patient
0,Rampur,Fever,Winter,2026,7,2,Paracetamol,1517,268,5.660448


In [13]:
required["Required_Quantity"]=(

    required["Predicted_Cases"]

    *

    required["Medicine_Per_Patient"]

)

required

,Village,Disease,Season,Year,Month,Predicted_Cases,Medicine_Name,Quantity,Patients,Medicine_Per_Patient,Required_Quantity
0,Rampur,Fever,Winter,2026,7,2,Paracetamol,1517,268,5.660448,11.320896


In [14]:
required = required.merge(

    medicine,

    on="Medicine_Name"

)

required

,Village,Disease,Season,Year,Month,Predicted_Cases,Medicine_Name,Quantity,Patients,Medicine_Per_Patient,Required_Quantity,Medicine_ID,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name
0,Rampur,Fever,Winter,2026,7,2,Paracetamol,1517,268,5.660448,11.320896,M001,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC


In [15]:
required["Remaining_Stock"]=(

    required["Current_Stock"]

    -

    required["Required_Quantity"]

)

required

,Village,Disease,Season,Year,Month,Predicted_Cases,Medicine_Name,Quantity,Patients,Medicine_Per_Patient,Required_Quantity,Medicine_ID,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name,Remaining_Stock
0,Rampur,Fever,Winter,2026,7,2,Paracetamol,1517,268,5.660448,11.320896,M001,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC,488.679104


In [16]:
def stock_alert(row):

    if row["Remaining_Stock"] <= 0:

        return "OUT OF STOCK"

    elif row["Remaining_Stock"] <= row["Reorder_Level"]:

        return "LOW STOCK"

    else:

        return "STOCK AVAILABLE"

In [17]:
required["Alert"] = required.apply(

    stock_alert,

    axis=1

)

required

,Village,Disease,Season,Year,Month,Predicted_Cases,Medicine_Name,Quantity,Patients,Medicine_Per_Patient,Required_Quantity,Medicine_ID,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name,Remaining_Stock,Alert
0,Rampur,Fever,Winter,2026,7,2,Paracetamol,1517,268,5.660448,11.320896,M001,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC,488.679104,STOCK AVAILABLE


In [18]:
medicine["Expiry_Date"] = pd.to_datetime(

    medicine["Expiry_Date"]

)

today = pd.Timestamp.today()

medicine["Days_Left"]=(

    medicine["Expiry_Date"]

    -

    today

).dt.days

In [19]:
medicine["Expiry_Alert"]=np.where(

    medicine["Days_Left"]<30,

    "EXPIRING SOON",

    "SAFE"

)

medicine

,Medicine_ID,Medicine_Name,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name,Days_Left,Expiry_Alert
0,M001,Paracetamol,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC,335,SAFE
1,M002,ORS,180,80,In Stock,2027-03-15,HealthMed,Rampur PHC,228,SAFE
2,M003,Cough Syrup,75,50,Low Stock,2026-11-10,MediCare,Shivpur PHC,103,SAFE
3,M004,Cetirizine,220,60,In Stock,2027-01-20,ABC Pharma,Shivpur PHC,174,SAFE
4,M005,Amoxicillin,95,100,Low Stock,2026-12-25,LifeCare,Ganeshwadi PHC,148,SAFE
5,M006,Azithromycin,140,70,In Stock,2027-05-18,Cure Pharma,Ganeshwadi PHC,292,SAFE
6,M007,Artemether,110,60,In Stock,2027-04-01,HealthMed,Lakshmi PHC,245,SAFE
7,M008,Rifampicin,85,50,In Stock,2027-08-10,TB Care,Lakshmi PHC,376,SAFE
8,M009,Salbutamol,40,50,Low Stock,2026-10-05,MediCare,Savitri PHC,67,SAFE
9,M010,Iron Tablets,260,100,In Stock,2027-07-15,NutriMed,Savitri PHC,350,SAFE


In [20]:
import os

os.makedirs("output", exist_ok=True)

required.to_csv(
    "medicine_demand_prediction.csv",
    index=False
)

print("Medicine Demand Prediction Saved Successfully")

Medicine Demand Prediction Saved Successfully


In [22]:
medicine["Expiry_Date"] = pd.to_datetime(medicine["Expiry_Date"])

In [23]:
today = pd.Timestamp.today()

medicine["Days_Left"] = (
    medicine["Expiry_Date"] - today
).dt.days

medicine.head()

,Medicine_ID,Medicine_Name,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name,Days_Left,Expiry_Alert
0,M001,Paracetamol,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC,335,SAFE
1,M002,ORS,180,80,In Stock,2027-03-15,HealthMed,Rampur PHC,228,SAFE
2,M003,Cough Syrup,75,50,Low Stock,2026-11-10,MediCare,Shivpur PHC,103,SAFE
3,M004,Cetirizine,220,60,In Stock,2027-01-20,ABC Pharma,Shivpur PHC,174,SAFE
4,M005,Amoxicillin,95,100,Low Stock,2026-12-25,LifeCare,Ganeshwadi PHC,148,SAFE


In [24]:
medicine["Expiry_Alert"] = np.where(
    medicine["Days_Left"] <= 30,
    "EXPIRING SOON",
    "SAFE"
)

medicine.head()

,Medicine_ID,Medicine_Name,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name,Days_Left,Expiry_Alert
0,M001,Paracetamol,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC,335,SAFE
1,M002,ORS,180,80,In Stock,2027-03-15,HealthMed,Rampur PHC,228,SAFE
2,M003,Cough Syrup,75,50,Low Stock,2026-11-10,MediCare,Shivpur PHC,103,SAFE
3,M004,Cetirizine,220,60,In Stock,2027-01-20,ABC Pharma,Shivpur PHC,174,SAFE
4,M005,Amoxicillin,95,100,Low Stock,2026-12-25,LifeCare,Ganeshwadi PHC,148,SAFE


In [25]:
stock_alert = medicine.copy()

stock_alert.head()

,Medicine_ID,Medicine_Name,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name,Days_Left,Expiry_Alert
0,M001,Paracetamol,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC,335,SAFE
1,M002,ORS,180,80,In Stock,2027-03-15,HealthMed,Rampur PHC,228,SAFE
2,M003,Cough Syrup,75,50,Low Stock,2026-11-10,MediCare,Shivpur PHC,103,SAFE
3,M004,Cetirizine,220,60,In Stock,2027-01-20,ABC Pharma,Shivpur PHC,174,SAFE
4,M005,Amoxicillin,95,100,Low Stock,2026-12-25,LifeCare,Ganeshwadi PHC,148,SAFE


In [26]:
stock_alert = stock_alert.merge(
    required[
        [
            "Medicine_Name",
            "Required_Quantity",
            "Remaining_Stock",
            "Alert"
        ]
    ],
    on="Medicine_Name",
    how="left"
)

stock_alert.head()

,Medicine_ID,Medicine_Name,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name,Days_Left,Expiry_Alert,Required_Quantity,Remaining_Stock,Alert
0,M001,Paracetamol,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC,335,SAFE,11.320896,488.679104,STOCK AVAILABLE
1,M002,ORS,180,80,In Stock,2027-03-15,HealthMed,Rampur PHC,228,SAFE,NaN,NaN,NaN
2,M003,Cough Syrup,75,50,Low Stock,2026-11-10,MediCare,Shivpur PHC,103,SAFE,NaN,NaN,NaN
3,M004,Cetirizine,220,60,In Stock,2027-01-20,ABC Pharma,Shivpur PHC,174,SAFE,NaN,NaN,NaN
4,M005,Amoxicillin,95,100,Low Stock,2026-12-25,LifeCare,Ganeshwadi PHC,148,SAFE,NaN,NaN,NaN


In [27]:
stock_alert["Required_Quantity"] = stock_alert["Required_Quantity"].fillna(0)

stock_alert["Remaining_Stock"] = stock_alert["Remaining_Stock"].fillna(
    stock_alert["Current_Stock"]
)

stock_alert["Alert"] = stock_alert["Alert"].fillna("NO DEMAND")

stock_alert.head()

,Medicine_ID,Medicine_Name,Current_Stock,Reorder_Level,Stock_Status,Expiry_Date,Supplier,PHC_Name,Days_Left,Expiry_Alert,Required_Quantity,Remaining_Stock,Alert
0,M001,Paracetamol,500,120,In Stock,2027-06-30,ABC Pharma,Rampur PHC,335,SAFE,11.320896,488.679104,STOCK AVAILABLE
1,M002,ORS,180,80,In Stock,2027-03-15,HealthMed,Rampur PHC,228,SAFE,0.000000,180.000000,NO DEMAND
2,M003,Cough Syrup,75,50,Low Stock,2026-11-10,MediCare,Shivpur PHC,103,SAFE,0.000000,75.000000,NO DEMAND
3,M004,Cetirizine,220,60,In Stock,2027-01-20,ABC Pharma,Shivpur PHC,174,SAFE,0.000000,220.000000,NO DEMAND
4,M005,Amoxicillin,95,100,Low Stock,2026-12-25,LifeCare,Ganeshwadi PHC,148,SAFE,0.000000,95.000000,NO DEMAND


In [28]:
import os

os.makedirs("output", exist_ok=True)

required.to_csv(
    "medicine_requirement.csv",
    index=False
)

stock_alert.to_csv(
    "medicine_stock_alert.csv",
    index=False
)

print("All files saved successfully!")

All files saved successfully!
